In [ ]:
# ============================================================
# ASSIGNMENT 6: OBJECT DETECTION USING TRANSFER LEARNING
# Model: YOLOv8n | Dataset: COCO128
# Includes freezing, fine-tuning, evaluation and visualization
# ============================================================

%pip install -q ultralytics matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from ultralytics import YOLO

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Set a relative base directory so it works in both Colab and local Jupyter
base_dir = Path("./assignment6_results").resolve()

In [ ]:
# ============================================================
# 1. LOAD PRETRAINED MODEL
# ============================================================
model = YOLO("yolov8n.pt")
print("Pretrained YOLOv8n model loaded.")

# ============================================================
# 2. STAGE 1: TRAIN WITH INITIAL LAYERS FROZEN
# ============================================================
print("\nStarting Stage 1: Frozen-layer training...")

model.train(
    data="coco128.yaml",
    epochs=5,
    imgsz=640,
    batch=8,
    freeze=10,
    device=device,
    workers=0,
    project=str(base_dir),
    name="stage1_frozen",
    exist_ok=True,
    plots=True
)

# Obtain the actual directory used by Ultralytics
stage1_dir = Path(model.trainer.save_dir)
stage1_path = stage1_dir / "weights" / "best.pt"

# Fall back to last.pt if best.pt is unavailable
if not stage1_path.exists():
    stage1_path = stage1_dir / "weights" / "last.pt"

if not stage1_path.exists():
    raise FileNotFoundError(f"No trained weights found in: {stage1_dir / 'weights'}")

print("Stage 1 completed.")
print("Stage 1 weights:", stage1_path)

In [ ]:
# ============================================================
# 3. STAGE 2: FINE-TUNE ALL LAYERS
# ============================================================
print("\nStarting Stage 2: Fine-tuning...")

model = YOLO(str(stage1_path))

model.train(
    data="coco128.yaml",
    epochs=5,
    imgsz=640,
    batch=8,
    freeze=0,
    lr0=0.001,
    device=device,
    workers=0,
    project=str(base_dir),
    name="stage2_finetune",
    exist_ok=True,
    plots=True
)

stage2_dir = Path(model.trainer.save_dir)
final_path = stage2_dir / "weights" / "best.pt"

if not final_path.exists():
    final_path = stage2_dir / "weights" / "last.pt"

if not final_path.exists():
    raise FileNotFoundError(f"No final weights found in: {stage2_dir / 'weights'}")

print("Fine-tuning completed.")
print("Final model:", final_path)

In [ ]:
# ============================================================
# 4. EVALUATE FINAL MODEL
# ============================================================
model = YOLO(str(final_path))
print("\nEvaluating final model...")

metrics = model.val(
    data="coco128.yaml",
    imgsz=640,
    batch=8,
    device=device,
    workers=0,
    project=str(base_dir),
    name="evaluation",
    exist_ok=True,
    plots=True
)

print("\n========== EVALUATION RESULTS ==========")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

In [ ]:
# ============================================================
# 5. OBJECT DETECTION AND BOUNDING-BOX VISUALIZATION
# ============================================================
print("\nRunning object detection on a sample image...")

image_url = "https://ultralytics.com/images/bus.jpg"

results = model.predict(
    source=image_url,
    conf=0.25,
    imgsz=640,
    device=device,
    save=False
)

# Draw bounding boxes, labels and confidence scores
annotated = results[0].plot()

# Convert BGR to RGB for Matplotlib
annotated = annotated[:, :, ::-1]

plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.title("Object Detection Using Transfer Learning")
plt.axis("off")
plt.tight_layout()

output_image = base_dir / "assignment6_object_detection.png"
plt.savefig(output_image, dpi=150, bbox_inches="tight")
plt.show()



In [ ]:
# ============================================================
# 6. DISPLAY DETECTED OBJECTS
# ============================================================
print("\n========== DETECTED OBJECTS ==========")
boxes = results[0].boxes

if boxes is not None and len(boxes) > 0:
    for box in boxes:
        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        coordinates = box.xyxy[0].tolist()
        print(
            f"Object: {model.names[class_id]} | "
            f"Confidence: {confidence:.2%} | "
            f"Bounding box: {[round(v, 2) for v in coordinates]}"
        )
else:
    print("No objects detected.")



In [ ]:
# ============================================================
# 7. FINAL OUTPUT
# ============================================================
print("\n========== ASSIGNMENT COMPLETED ==========")
print("Stage 1: Frozen-layer training")
print("Stage 2: Fine-tuning")
print("Final model:", final_path)
print("Detection image:", output_image)
print("Results directory:", stage2_dir)
print("Evaluation metrics and plots have been saved.")